In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
actual = pd.read_csv('data/actual.csv')
test_df = pd.read_csv('data/data_set_ALL_AML_independent.csv')
train_df = pd.read_csv('data/data_set_ALL_AML_train.csv')

In [3]:
df1 = [col for col in train_df.columns if "call" not in col]
train_df = train_df[df1]
train_df.T.head()
train_df = train_df.T
train_df2 = train_df.drop(['Gene Description','Gene Accession Number'],axis=0)
train_df2.index = pd.to_numeric(train_df2.index)
train_df2.sort_index(inplace=True)
train_df2.head()

,0,1,2,3,4,5,6,7,8,9,...,7119,7120,7121,7122,7123,7124,7125,7126,7127,7128
1,-214,-153,-58,88,-295,-558,199,-176,252,206,...,185,511,-125,389,-37,793,329,36,191,-37
2,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,169,837,-36,442,-17,782,295,11,76,-14
3,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,315,1199,33,168,52,1138,777,41,228,-41
4,-135,-114,265,12,-419,-585,158,-253,49,31,...,240,835,218,174,-110,627,170,-50,126,-91
5,-106,-125,-76,168,-230,-284,4,-122,70,252,...,156,649,57,504,-26,250,314,14,56,-25


In [4]:
train_df2['cancer_type'] = list(pd.read_csv('data/actual.csv')[:38]['cancer'])
dic = {'ALL':0,'AML':1}
train_df2.replace(dic,inplace=True)

/tmp/ipykernel_5645/3243176898.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df2.replace(dic,inplace=True)


Sparse Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(train_df2.drop('cancer_type',axis=1))
y = train_df2['cancer_type']

In [12]:
logreg = LogisticRegression(penalty='l1', solver='saga', random_state=0, C=0.08).fit(X_std, y)
logreg.predict(X_std)

score = logreg.score(X_std, y)
print('score:',score)
coef = logreg.coef_.ravel()
nonzero_idx = np.where(coef != 0)[0]
print('nonzero coefs:', len(nonzero_idx))

score: 0.8947368421052632
nonzero coefs: 10


/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [10]:
df1 = [col for col in test_df.columns if "call" not in col]
test_df = test_df[df1]
test_df = test_df.T
test_df2 = test_df.drop(['Gene Description','Gene Accession Number'],axis=0)
test_df2.index = pd.to_numeric(test_df2.index)
test_df2.sort_index(inplace=True)
test_df2['cancer_type'] = list(pd.read_csv('data/actual.csv')[38:]['cancer'])
dic = {'ALL':0,'AML':1}
test_df2.replace(dic,inplace=True)
test_df2.head()

TypeError: argument of type 'int' is not iterable

In [13]:
X_test_scaled = StandardScaler().fit_transform(test_df2.drop('cancer_type',axis=1))
y_test = test_df2['cancer_type']
test_score = logreg.score(X_test_scaled,y_test)
print('test score:', test_score)

test score: 0.7058823529411765
